# Review Star Rater: QLoRA fine-tuning of Qwen2.5-0.5B
Yelp review → star rating (1–5). **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

You will be asked twice: **Google Drive access** (Step 2) and your **Hugging Face Write token** (Step 2, only if there is no `HF_TOKEN` secret).

What's fixed compared with the first run:
- T4 has no real bfloat16, so everything uses **float32**. No more dtype-mismatch errors or hook workarounds.
- The adapter is **saved to Google Drive during and after training, and pushed to Hugging Face**, so it can't be lost when the runtime resets.
- At the end the saved adapter is **reloaded from Drive and tested**, which proves it was saved.

## Step 1: Setup

In [ ]:
import subprocess
try:
    gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
except FileNotFoundError:            # CPU runtime: nvidia-smi doesn't exist
    gpu = subprocess.CompletedProcess([], 1, "", "")
if gpu.returncode != 0:
    raise SystemExit("NO GPU: Runtime → Change runtime type → T4 GPU, then Runtime → Run all again.")
print(gpu.stdout.split("\n")[8] if len(gpu.stdout.split("\n")) > 8 else gpu.stdout)

In [ ]:
!pip uninstall -y -q diffusers gradio gradio-client
!pip install -q bitsandbytes==0.48.2 trl==0.25.1 transformers==4.57.6 peft datasets accelerate

## Step 2: Imports + Settings + Hugging Face Login

In [ ]:
import os, re, json
from collections import Counter
import numpy as np
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Model and data
BASE_MODEL = "Qwen/Qwen2.5-0.5B"
DATASET_NAME = "Yelp/yelp_review_full"
SEED = 42
TRAIN_SIZE = 2000
VALIDATION_SIZE = 200
TEST_SIZE = 300
MAX_REVIEW_CHARS = 600

# Training
NUM_EPOCHS = 1
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
MAX_LENGTH = 192
LEARNING_RATE = 2e-4
LR_SCHEDULER_TYPE = "cosine"
WARMUP_RATIO = 0.03
OPTIMIZER = "paged_adamw_32bit"
EVAL_STEPS = 50

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Precision: T4 has no real bfloat16 -> float32 everywhere (0.5B fits easily)
COMPUTE_DTYPE = "float32"

# Where the adapter is saved (Drive + Hugging Face)
HF_USER = "shivkrchy9876"
RUN_NAME = "review-star-rater-qlora"
SAVE_DIR = f"/content/drive/MyDrive/review_star_rater/{RUN_NAME}"
HF_REPO = f"{HF_USER}/{RUN_NAME}"

set_seed(SEED)
print("Effective batch size:", BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
print("Adapter will be saved to:", SAVE_DIR, "and", HF_REPO)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(SAVE_DIR, exist_ok=True)
print("Drive ready:", SAVE_DIR)

In [ ]:
from huggingface_hub import login, whoami
try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
except Exception:
    login()   # paste a WRITE token from huggingface.co/settings/tokens

info = whoami()
role = info.get("auth", {}).get("accessToken", {}).get("role")
print("Hugging Face user:", info["name"], "| token role:", role)
PUSH_TO_HUB = role == "write" and info["name"] == HF_USER
if not PUSH_TO_HUB:
    # Stop BEFORE training, so no training time is wasted on an adapter that can't be uploaded
    raise SystemExit(f"STOP: this token can't upload to {HF_USER} (user={info['name']}, role={role}). "
                     "Make a WRITE token at huggingface.co/settings/tokens, re-run this cell, then Run after.")
print("✅ Write token OK - the adapter will be uploaded to", HF_REPO)

## Step 3: Data

In [ ]:
dataset = load_dataset(DATASET_NAME)
train_pool = dataset["train"].shuffle(seed=SEED)
test_pool = dataset["test"].shuffle(seed=SEED)

train_data = train_pool.select(range(TRAIN_SIZE))
validation_data = train_pool.select(range(TRAIN_SIZE, TRAIN_SIZE + VALIDATION_SIZE))
test_data = test_pool.select(range(TEST_SIZE))

for name, d in [("Train", train_data), ("Validation", validation_data), ("Test", test_data)]:
    counts = Counter(x + 1 for x in d["label"])
    print(f"{name}: {len(d)} reviews, stars {dict(sorted(counts.items()))}")

overlap = len(set(train_data["text"]) & set(test_data["text"]))
print("Train-test overlap:", overlap)

In [ ]:
QUESTION = "How many stars (1-5) did the reviewer give?"

def make_prompt(text):
    return f"Review: {text[:MAX_REVIEW_CHARS]}\n\n{QUESTION}\nStars:"

def format_review(example):
    return {"prompt": make_prompt(example["text"]), "completion": f" {example['label'] + 1}"}

# keep only prompt/completion columns for the trainer
train_formatted = train_data.map(format_review, remove_columns=train_data.column_names)
validation_formatted = validation_data.map(format_review, remove_columns=validation_data.column_names)

print(train_formatted[0]["prompt"][-200:])
print("Completion:", repr(train_formatted[0]["completion"]))

## Step 4: Load the 4-bit model

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float32,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quantization_config,
    dtype=torch.float32,          # embeddings / lm_head in float32 too -> no dtype mismatch
    device_map="auto",
)
print(f"Loaded {BASE_MODEL}: {model.get_memory_footprint() / 1e6:.0f} MB on {model.device}")

## Step 5: Baseline evaluation

In [ ]:
def predict_stars(m, review_text):
    inputs = tokenizer(make_prompt(review_text), return_tensors="pt",
                       truncation=True, max_length=MAX_LENGTH).to(m.device)
    with torch.no_grad():
        out = m.generate(**inputs, max_new_tokens=3, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id)
    answer = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    match = re.match(r"^[1-5]", answer)
    return int(match.group()) if match else None

def evaluate(m, data, name):
    m.eval()
    truth, preds = [], []
    for ex in data:
        truth.append(ex["label"] + 1)
        preds.append(predict_stars(m, ex["text"]))
    truth = np.array(truth)
    guess = np.array([p if p is not None else 3 for p in preds])   # unparseable counts as 3
    r = {
        "accuracy": float(np.mean(guess == truth)),
        "off_by_one": float(np.mean(np.abs(guess - truth) <= 1)),
        "mae": float(np.mean(np.abs(guess - truth))),
        "unparseable": sum(p is None for p in preds) / len(preds),
    }
    print(f"RESULT model={name} accuracy={r['accuracy']:.3f} off_by_one={r['off_by_one']:.3f} "
          f"mae={r['mae']:.3f} unparseable={r['unparseable']:.3f}")
    return r, truth, guess

baseline, _, _ = evaluate(model, test_data, "base")

## Step 6: QLoRA training (adapter saved to Drive every 50 steps)

In [ ]:
model = prepare_model_for_kbit_training(model)

qlora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

training_config = SFTConfig(
    output_dir=SAVE_DIR,                 # checkpoints go straight to Google Drive
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    warmup_ratio=WARMUP_RATIO,
    optim=OPTIMIZER,
    max_length=MAX_LENGTH,
    completion_only_loss=True,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    logging_steps=10,
    save_strategy="steps",
    save_steps=EVAL_STEPS,
    save_total_limit=2,
    bf16=False,
    fp16=False,                          # float32 on T4: stable, no nan
    report_to="none",
    seed=SEED,
    push_to_hub=PUSH_TO_HUB,
    hub_model_id=HF_REPO,
    hub_private_repo=True,
    hub_strategy="every_save",
)

trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=train_formatted,
    eval_dataset=validation_formatted,
    peft_config=qlora_config,
    processing_class=tokenizer,
)
trainer.model.print_trainable_parameters()

In [ ]:
training_results = trainer.train()
print("Training loss:", training_results.training_loss, "| steps:", training_results.global_step)

## Step 7: SAVE the adapter (Drive + Hugging Face)

In [ ]:
trainer.save_model(SAVE_DIR)              # adapter_config.json + adapter_model.safetensors
tokenizer.save_pretrained(SAVE_DIR)
with open(os.path.join(SAVE_DIR, "training_settings.json"), "w") as f:
    json.dump({k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, list))}, f, indent=2)

saved = sorted(os.listdir(SAVE_DIR))
print("1) Saved in Google Drive:", SAVE_DIR)
print("  ", saved)
assert "adapter_model.safetensors" in saved, "Adapter file missing in Drive!"

In [ ]:
# 2) Upload the final adapter + tokenizer + settings to your Hugging Face repo, then CHECK it arrived
from huggingface_hub import HfApi, create_repo

api = HfApi()
create_repo(HF_REPO, private=True, exist_ok=True)
api.upload_folder(
    folder_path=SAVE_DIR,
    repo_id=HF_REPO,
    ignore_patterns=["checkpoint-*", "runs/*"],
    commit_message="Final QLoRA adapter, tokenizer and settings",
)
hub_files = api.list_repo_files(HF_REPO)
print("2) Files on Hugging Face:", hub_files)
assert "adapter_model.safetensors" in hub_files, "Upload failed - adapter not on Hugging Face!"
print("✅ Adapter is on Hugging Face:", f"https://huggingface.co/{HF_REPO}")

In [ ]:
# 3) A zip copy downloaded to your Mac as a third backup
import shutil
from google.colab import files
zip_path = shutil.make_archive(f"/content/{RUN_NAME}", "zip", SAVE_DIR)
print("3) Downloading", zip_path)
files.download(zip_path)

## Step 8: Evaluate the fine-tuned model

In [ ]:
ft_model = trainer.model
ft_model.gradient_checkpointing_disable()
ft_model.config.use_cache = True

tuned, truth, tuned_guess = evaluate(ft_model, test_data, "fine-tuned")

print("\nMetric        Base    Fine-tuned")
for k in ["accuracy", "off_by_one", "mae", "unparseable"]:
    print(f"{k:<12} {baseline[k]:.3f}   {tuned[k]:.3f}")

with open(os.path.join(SAVE_DIR, "results.json"), "w") as f:
    json.dump({"base": baseline, "fine_tuned": tuned}, f, indent=2)
api.upload_file(path_or_fileobj=os.path.join(SAVE_DIR, "results.json"), path_in_repo="results.json",
                repo_id=HF_REPO, commit_message="Evaluation results")
print("Results saved to Drive and Hugging Face")

In [ ]:
cm = confusion_matrix(truth, tuned_guess, labels=[1, 2, 3, 4, 5])
ConfusionMatrixDisplay(cm, display_labels=[1, 2, 3, 4, 5]).plot(values_format="d")
plt.title("QLoRA model: confusion matrix"); plt.xlabel("Predicted stars"); plt.ylabel("Actual stars")
plt.show()

## Step 9: Proof the adapter is saved: reload it from Drive and test

In [ ]:
del trainer, ft_model, model
import gc; gc.collect(); torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=quantization_config, dtype=torch.float32, device_map="auto")
reloaded = PeftModel.from_pretrained(base, SAVE_DIR)

review = test_data[0]["text"]
print("Actual stars:", test_data[0]["label"] + 1, "| reloaded model predicts:", predict_stars(reloaded, review))
print("Adapter reloaded from Drive successfully. Next time load it with:")
print(f"PeftModel.from_pretrained(base, '{SAVE_DIR}')  or  PeftModel.from_pretrained(base, '{HF_REPO}')")